# LightGBM と早期終了

**出典**　機械学習 4 分野の教科書 — 第 4 章 1-C アンサンブル・評価・前処理 ／ 03 勾配ブースティング：表形式データの現在の王者

**このノートのパス**　`ml-code/04_ensemble_eval/02_gbdt_early_stopping.ipynb`

**このノートで確かめること**
**早期終了には検証用データが必須**であること、`num_boost_round=5000` と置いても
実際には数百本で止まることを確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `lightgbm, scikit-learn` |
| 実行時間の目安 | 約 20 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q lightgbm scikit-learn

In [3]:
import lightgbm as lgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=0)

# 早期終了には検証用データが要る。訓練データをさらに分けて作る
# （テストデータ X_te は最後の 1 回しか触らない——触ると評価が甘くなる）
X_fit, X_va, y_fit, y_va = train_test_split(
    X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=0)

train_set = lgb.Dataset(X_fit, y_fit)
valid_set = lgb.Dataset(X_va, y_va, reference=train_set)

params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,     # 小さめにして本数で稼ぐのが定石
    "num_leaves": 31,          # 木の複雑さ（LightGBM は深さではなく葉数で制御する）
    "feature_fraction": 0.8,   # 各木で使う特徴量の割合（ランダムフォレストの工夫②に相当）
    "bagging_fraction": 0.8,   # 各木で使うデータの割合
    "bagging_freq": 1,
    "verbosity": -1,
}

model = lgb.train(
    params, train_set,
    num_boost_round=5000,
    valid_sets=[valid_set],
    callbacks=[
        lgb.early_stopping(100),   # ⚠️ 必須。検証スコアが 100 回改善しなければ打ち切る
        lgb.log_evaluation(200),
    ],
)
print("最良の木の本数:", model.best_iteration)
print("AUC:", roc_auc_score(y_te, model.predict(X_te, num_iteration=model.best_iteration)))

Training until validation scores don't improve for 100 rounds


[200]	valid_0's auc: 0.998452


[400]	valid_0's auc: 0.999484
Early stopping, best iteration is:
[328]	valid_0's auc: 0.999484
最良の木の本数: 328
AUC: 0.9861111111111112


### 早期終了を外すとどうなるか

In [4]:
no_stop = lgb.train(params, train_set, num_boost_round=1500, valid_sets=[valid_set],
                    callbacks=[lgb.log_evaluation(0)])
print("早期終了あり:", model.best_iteration, "本 / AUC",
      round(roc_auc_score(y_te, model.predict(X_te, num_iteration=model.best_iteration)), 4))
print("早期終了なし: 1500 本 / AUC",
      round(roc_auc_score(y_te, no_stop.predict(X_te)), 4))

早期終了あり: 328 本 / AUC 0.9861
早期終了なし: 1500 本 / AUC 0.9881


## 読み取りどころ

- **`num_boost_round=5000` は上限であって、実際に使う本数ではない。**早期終了が本数を決める。
- **早期終了は「1 つの学習の中で木の本数を打ち切る」仕組み**である。1-D で扱う **Pruning（試行そのものを捨てる枝刈り）とは別物**で、両方あって初めて探索が速くなる。
- ⚠️ **早期終了用の検証データを、閾値決定や最終評価に使い回さないこと。**そのデータに対して「最も良いラウンド」を選んでいるので、すでに選択が入っている。